# DSL on political ad tone

Re-run the Egami et al. test on Fowler et al.'s ads: **are Facebook ads less negative than TV ads?**

1. Get the true answer from expert labels
2. Have an LLM label every ad
3. Pretend only a few ads have expert labels
4. Estimate the effect three ways
5. Repeat many times and see which method finds the truth

In [ ]:
from dsl_demo import load_ads, llm_labels, ols, draw   # first: imports torch before scikit-learn
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold

MODEL     = "Qwen/Qwen2.5-0.5B-Instruct"   # small, fast; bigger models are not better with this prompt
MAX_CHARS = 600     # characters of ad text shown to the LLM
N_CORPUS  = 1000    # ads in our corpus
N_EXPERT  = 75      # ads with expert labels (~7.5%, like the paper)
REPEATS   = 200
SEED      = 6870

## 1. The true answer (all expert labels)

In [ ]:
ads = load_ads(N_CORPUS, SEED)
TRUTH, _ = ols(ads.Y, ads.X)
print(ads.groupby("platform").Y.mean().round(3))
print(f"True Facebook effect: {TRUTH:.3f}")

## 2. An open-source LLM labels every ad

The labels for the default model and prompt are saved in `data/`, so this is instant. Change the prompt or model and it runs the LLM (see `dsl_demo.py`).

In [ ]:
SYSTEM = (
    "You code political ads. Is the primary purpose of the ad to promote a specific candidate, "
    "attack a candidate, or contrast the candidates?\n"
    "Promote: only talks about the sponsoring candidate.\n"
    "Contrast: talks about both the sponsor and the opponent.\n"
    "Attack: only talks about the opponent.\n"
    "Answer with one word: Promote, Contrast, or Attack."
)
ads["Q_tone"] = llm_labels(ads, MODEL, SYSTEM, MAX_CHARS)   # saved labels if we have them, else runs the LLM
ads["Q"] = ads.Q_tone.isin(["Attack", "Contrast"]).astype(int)
print("LLM agrees with experts:", round((ads.Q == ads.Y).mean(), 3))
print(ads.assign(agree=ads.Q == ads.Y).groupby("platform").agree.mean().round(3))

Where is the LLM wrong more often: Facebook or TV? Why does that matter for the regression?

## 3. Pretend only a few ads have expert labels

In [ ]:
PI = N_EXPERT / N_CORPUS          # every ad has the same, known chance of being picked

## 4. Three ways to estimate the effect

In [ ]:
def llm_only(R):                       # just trust the LLM labels
    return ols(ads.Q, ads.X)

T = TfidfVectorizer(min_df=2, max_features=5000).fit_transform(ads.text)   # word features, built once

def supervised(R):                     # train a text classifier on the expert labels, predict the rest
    lab = R == 1
    clf = LogisticRegression(max_iter=2000).fit(T[lab], ads.Y[lab])
    y = np.where(lab, ads.Y, clf.predict_proba(T)[:, 1])
    return ols(y, ads.X)

G = pd.get_dummies(ads.Q_tone).astype(float)   # what DSL's correction model sees
G["X"], G["len"] = ads.X, np.log(ads.n_words)
G["QX"] = ads.Q * ads.X

def dsl(R, k=5, seed=0):               # LLM label + random expert checks + correction
    g = np.zeros(N_CORPUS)
    for train, test in KFold(k, shuffle=True, random_state=seed).split(G):
        tr = train[R[train] == 1]
        g[test] = LogisticRegression(max_iter=2000).fit(G.iloc[tr], ads.Y[tr]).predict_proba(G.iloc[test])[:, 1]
    y_tilde = g + R / PI * (ads.Y - g)  # the correction
    return ols(y_tilde, ads.X)

R = draw(N_CORPUS, N_EXPERT, 1)
for name, f in [("LLM only", llm_only), ("Supervised", supervised), ("DSL", dsl)]:
    b, se = f(R)
    print(f"{name:11s} {b:6.3f}  [{b - 1.96*se:6.3f}, {b + 1.96*se:6.3f}]")
print(f"{'Truth':11s} {TRUTH:6.3f}")

## 5. Repeat many times

In [ ]:
def simulate(methods):
    rows = []
    for r in range(REPEATS):
        R = draw(N_CORPUS, N_EXPERT, r)
        for name, f in methods:
            b, se = f(R)
            rows.append((name, b, se, abs(b - TRUTH) <= 1.96 * se))
    res = pd.DataFrame(rows, columns=["method", "estimate", "se", "covers_truth"])
    summary = res.groupby("method", sort=False).agg(mean_estimate=("estimate", "mean"),
                                                    coverage=("covers_truth", "mean"),
                                                    mean_se=("se", "mean")).round(3)
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
    for name, d in res.groupby("method", sort=False):
        ax[0].hist(d.estimate, bins=25, alpha=.6, label=name)
    ax[0].axvline(TRUTH, color="k", ls="--", label="Truth"); ax[0].set_title("Estimates"); ax[0].legend()
    summary.coverage.plot.bar(ax=ax[1], rot=0); ax[1].axhline(.95, color="k", ls="--")
    ax[1].set_ylim(0, 1); ax[1].set_title("How often the 95% interval covers the truth")
    plt.tight_layout()
    return summary

simulate([("LLM only", llm_only), ("Supervised", supervised), ("DSL", dsl)])

**Try:** change `N_EXPERT` to 40 or 150, swap `MODEL`, or add example ads to the prompt (few-shot). Which results change, and which don't?